# Investigando CVLI no Ceará (2009–2025)
## Análise Exploratória Descritiva

**Objetivo:** Construir uma base analítica sólida que descreva padrões, tendências e distribuições dos Crimes Violentos Letais Intencionais (CVLI) no Ceará.

**Variável target futura:** `total_cvli_ano` — contagem de CVLI por município/ano
**Estrutura:** Módulos integrados em `src/` e arquivos de dados em `data/`

---

### 0.1 Instalação de dependências (se necessário)

In [ ]:
!pip install geobr geopandas matplotlib seaborn pandas openpyxl

### 0.2 Importação de bibliotecas e módulos do projeto (src)

In [ ]:
import sys
import os

# Permitir importar pacotes da pasta raiz (src)
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))

import pandas as pd
import geobr as gpd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data import (
    load_cvli_data,
    load_planning_regions,
    load_municipality_geodata,
    clean_column_names,
    merge_geodata_and_regions
)
from src.features import (
    extract_temporal_features,
    create_age_groups,
    categorize_rmf_vs_interior
)
from src.visualization import (
    set_chart_style,
    plot_temporal_trend,
    plot_top_municipalities,
    plot_monthly_heatmap
)

set_chart_style()


### 0.3 Carregamento e limpeza dos dados

In [ ]:
# Carregamento via src.data
df = load_cvli_data()
planejamento = load_planning_regions()

# Padronização de colunas
df = clean_column_names(df)

print(f"CVLI: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Planejamento: {planejamento.shape[0]} linhas x {planejamento.shape[1]} colunas")
display(df.head())


### 0.4 Inspeção Inicial e Estatísticas Descritivas

In [ ]:
print("=" * 60)
print("INSPEÇÃO INICIAL")
print("=" * 60)

print(f"\nTipos de dados:")
print(df.dtypes)

print("\nVariáveis numéricas:")
display(df.describe())

print("\nVariáveis categóricas:")
display(df.describe(include="object"))


### 0.5 Integração Geográfica e Construção do Painel Agregado

In [ ]:
# Geobr
geo_ce = load_municipality_geodata()

# Agregado por município e ano (Target: total_cvli_ano)
agregado = df.groupby(['município', 'ano']).size().reset_index(name='total_cvli_ano')

# Merge de coordenadas e regiões via src.data
agregado = merge_geodata_and_regions(agregado, geo_ce, planejamento)

print("Estrutura do painel agregado:")
print(agregado.dtypes)
display(agregado.head())


---
## Seção 1 — Diagnóstico de Qualidade dos Dados (Skill Análise)

In [ ]:
# Diagnóstico de Missings
missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
display(missing.sort_values("missing_pct", ascending=False))

for col in ['idade_da_vítima', 'escolaridade_da_vítima', 'raça_da_vítima']:
    if col in df.columns:
        nao_inf = (df[col] == 'Não Informada').sum()
        pct = (nao_inf / len(df) * 100).round(2)
        print(f"{col} - 'Não Informada': {nao_inf:,} ({pct}%)")


---
## Seção 2 — Evolução Temporal do CVLI (2009–2025)

In [ ]:
df_temporal = df.groupby('ano').size().reset_index(name='total_cvli')
df_temporal['variacao_pct'] = (df_temporal['total_cvli'].pct_change() * 100).round(2)
display(df_temporal)

fig, ax = plot_temporal_trend(df_temporal)
plt.show()


---
## Seção 3 — Distribuição Geográfica

In [ ]:
ranking_muni = df.groupby('município').size().reset_index(name='total_cvli')
ranking_muni = ranking_muni.sort_values('total_cvli', ascending=False).reset_index(drop=True)
ranking_muni['pct_total'] = (ranking_muni['total_cvli'] / ranking_muni['total_cvli'].sum() * 100).round(2)

display(ranking_muni.head(20))

fig, ax = plot_top_municipalities(ranking_muni, top_n=20)
plt.show()


---
## Seção 4 — Análise por Natureza do Crime

In [ ]:
dist_natureza = df['natureza'].value_counts().reset_index()
dist_natureza.columns = ['natureza', 'total']
dist_natureza['pct'] = (dist_natureza['total'] / dist_natureza['total'].sum() * 100).round(2)
display(dist_natureza)

evo_natureza = df.groupby(['ano', 'natureza']).size().reset_index(name='total')
display(evo_natureza.pivot(index='ano', columns='natureza', values='total').fillna(0).astype(int))


---
## Seção 5 — Perfil das Vítimas (Subconjunto Informado)

In [ ]:
df_feat = create_age_groups(df)
if 'faixa_etaria' in df_feat.columns:
    dist_idade = df_feat['faixa_etaria'].value_counts().sort_index().reset_index()
    dist_idade.columns = ['faixa_etaria', 'total']
    display(dist_idade)


---
## Seção 6 — Fortaleza / RMF vs. Interior

In [ ]:
agregado_grupo = categorize_rmf_vs_interior(agregado)
comp_grupo = agregado_grupo.groupby('grupo')['total_cvli_ano'].sum().reset_index()
comp_grupo['pct'] = (comp_grupo['total_cvli_ano'] / comp_grupo['total_cvli_ano'].sum() * 100).round(2)
display(comp_grupo)


---
## Seção 7 — Análise de Sazonalidade

In [ ]:
df_temp_feat = extract_temporal_features(df)
heatmap_data = df_temp_feat.groupby(['ano', 'mes']).size().reset_index(name='total')
heatmap_pivot = heatmap_data.pivot(index='mes', columns='ano', values='total').fillna(0)

meses_nome = {1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai', 6:'Jun',
              7:'Jul', 8:'Ago', 9:'Set', 10:'Out', 11:'Nov', 12:'Dez'}
heatmap_pivot.index = [meses_nome[m] for m in heatmap_pivot.index]

fig, ax = plot_monthly_heatmap(heatmap_pivot)
plt.show()


---
## Seção 8 — Resumo Técnico e Próximos Passos

In [ ]:
print("============================================================")
print("RESUMO TÉCNICO E PRÓXIMOS PASSOS PARA MODELAGEM ESPACIAL / ML")
print("============================================================")
print(f"Total de registros analisados: {len(df):,}")
print(f"Total de municípios no painel: {agregado['município'].nunique()}")
print(f"Target configurado: total_cvli_ano")
print("Módulos em src/ prontos para expansão e treinamento de modelos.")
